# Sleeplens low-RAM cloud render

Run the full sleeplens pipeline on a free Google Colab T4 GPU. This is
the right path when your local machine runs out of memory during the
final encode: a 1080p H.264 video needs ~700 MB of free RAM for the
libx264 medium preset, and the per-pixel `geq` progress bar adds ~150
MB on top. Colab gives you 12.7 GB of system RAM plus a real NVIDIA
NVENC encoder, so the encode finishes in a couple of minutes.

**How to use:**
1. Click *Runtime -> Run all* (or press `Ctrl+F9`).
2. The upload cell will ask for your script text file and your
   background image - pick both files.
3. Wait for the render cell to finish (TTS, mix, encode).
4. The download cell will save the final MP4 to your machine.

**Cost:** free. Colab sessions cap at 12 hours and may disconnect if
idle, but a 6-minute video finishes in well under 10 minutes.

In [ ]:
# 1. Install sleeplens and its dependencies. Pulls from the public
# GitHub repo, so this only needs an internet connection.
!pip install -q "sleeplens @ git+https://github.com/fernandojjq/sleeplens.git"

# Colab's system ffmpeg already includes NVENC and CUDA; no need to
# apt-get install a custom build. We still probe it to confirm.
import subprocess
ffmpeg_version = subprocess.run(
    ["ffmpeg", "-version"], capture_output=True, text=True
).stdout.splitlines()[0]
print(ffmpeg_version)

nvenc = subprocess.run(
    ["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True
).stdout
print("NVENC available:" if "h264_nvenc" in nvenc else "NVENC NOT available - falling back to libx264")

In [ ]:
# 2a. (Optional) Generate the script in Colab from a topic.
# Skip this cell entirely if you already have a script file - the
# next cell will pick it up from the upload widget.
#
# The script uses NVIDIA NIM with DeepSeek V4 Flash by default
# (free tier, no credit card). Swap base_url + model for OpenAI,
# Anthropic-via-proxy, or any other OpenAI-compatible endpoint.
import os

GENERATE_SCRIPT = False  # flip to True to enable in-Colab script generation
TOPIC = ""  # e.g. "how transformers attend to context"
TARGET_WORDS = 4500
API_KEY = ""  # your NVIDIA NIM key (or OpenAI key, or ...)
BASE_URL = "https://integrate.api.nvidia.com/v1"
MODEL = "deepseek-ai/deepseek-v4-flash"

if GENERATE_SCRIPT:
    if not TOPIC or not API_KEY:
        raise SystemExit("Set TOPIC and API_KEY before flipping GENERATE_SCRIPT to True.")
    from openai import OpenAI
    client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=180.0)
    from sleeplens.ai.script_writer import SYSTEM_PROMPT
    user_prompt = f"Topic: {TOPIC}\nTarget word count: {TARGET_WORDS}."
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.7, max_tokens=8192,
    )
    generated = response.choices[0].message.content
    os.makedirs("/content/assets", exist_ok=True)
    safe_topic = TOPIC.replace(" ", "_")[:40]
    SCRIPT = f"/content/assets/{safe_topic}.txt"
    with open(SCRIPT, "w", encoding="utf-8") as fh:
        fh.write(generated)
    print(f"Generated {len(generated.split())} words -> {SCRIPT}")
else:
    print("Skipped. Upload a .txt file in the next cell.")

In [ ]:
# 2b. Upload your script text file and background image OR video.
# Both are sent through the browser, stay on Colab's VM, and are
# deleted when the session ends. Nothing reaches GitHub or any
# third party.
from google.colab import files
import os

os.makedirs("/content/assets", exist_ok=True)
os.makedirs("/content/output", exist_ok=True)

print("Upload your script text file (.txt) and background image or video...")
uploaded = files.upload()
for name, data in uploaded.items():
    with open(f"/content/assets/{name}", "wb") as fh:
        fh.write(data)
print(f"Uploaded: {sorted(uploaded.keys())}")

# Heuristics: pick the first .txt as the script, the first
# image-like file as a still background, or the first video as
# a loopable background. The render command picks whichever is set.
import glob
SCRIPT = ""
for pat in ("/content/assets/*.txt",):
    matches = sorted(glob.glob(pat))
    if matches:
        SCRIPT = matches[0]
        break

BG_IMAGE = ""
BG_VIDEO = ""
for pat in ("/content/assets/*.mp4", "/content/assets/*.mov", "/content/assets/*.webm"):
    matches = sorted(glob.glob(pat))
    if matches:
        BG_VIDEO = matches[0]
        break
if not BG_VIDEO:
    for pat in ("/content/assets/*.jpg", "/content/assets/*.jpeg", "/content/assets/*.png"):
        matches = sorted(glob.glob(pat))
        if matches:
            BG_IMAGE = matches[0]
            break

print(f"Script:      {SCRIPT or "<missing>"}")
print(f"Background:  {BG_IMAGE or BG_VIDEO or "<missing>"}")
print(f"  is video:  {bool(BG_VIDEO)}")

if not SCRIPT:
    raise SystemExit("Need a .txt script. Either run the cell above to generate one, or upload a .txt here.")
if not (BG_IMAGE or BG_VIDEO):
    raise SystemExit("Need a background image (.jpg/.png) or a video (.mp4/.mov).")

In [ ]:
# 3. Render the video. This runs the same sleeplens pipeline as the
# local CLI: TTS via Edge, mix with the procedural ambient bed, and
# the final ffmpeg encode. On Colab, the encode uses real NVENC, so
# a 6-minute 1080p video finishes in about 1-2 minutes.
import subprocess, sys

OUT_STEM = "sleeplens-colab"
cmd = [
    "python", "-m", "sleeplens", "render",
    "--script", SCRIPT,
    "--output-stem", OUT_STEM,
]
if BG_IMAGE:
    cmd += ["--background-image", BG_IMAGE]
if BG_VIDEO:
    cmd += ["--background-video", BG_VIDEO]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print("--- stdout (last 2KB) ---")
print(result.stdout[-2000:])
print("--- stderr (last 2KB) ---")
print(result.stderr[-2000:])
if result.returncode != 0:
    raise SystemExit(f"Render failed with exit code {result.returncode}")

In [ ]:
# 4. Download the final MP4. Colab saves it to /content/output/ and
# hands it to the browser, which triggers a normal download.
from google.colab import files
import glob, os

candidates = sorted(glob.glob(f"/content/output/{OUT_STEM}.mp4"))
if not candidates:
    raise SystemExit("No MP4 was produced. Check the render cell for errors.")
output = candidates[0]
print(f"Final video: {output} ({os.path.getsize(output)/1e6:.1f} MB)")
files.download(output)

## What just happened

- **Cell 1** installed sleeplens from the public repo and confirmed
  Colab's ffmpeg build exposes NVENC (it always does on T4 instances).
- **Cell 2a** optionally generated the script in Colab from a topic
  using whatever API key you pasted in. Skip it if you uploaded a
  pre-written script.
- **Cell 2b** took the files you uploaded and resolved the script path,
  the background image, and the background video (whichever you gave).
  Either an image or a short loopable video works.
- **Cell 3** ran the full sleeplens CLI: TTS via Edge, mix with the
  procedural ambient bed, encode with `h264_nvenc` on the T4.
- **Cell 4** downloaded the MP4 to your machine.

## Why is the render taking longer than 1-2 minutes?

A 6-minute 1080p video should finish in **under 5 minutes** with the
T4 GPU. If it takes 10-15 minutes, one of these is happening:

- **GPU not enabled.** Open *Runtime -> Change runtime type* and pick
  *T4 GPU* under Hardware accelerator. This is the most common cause.
  Without GPU, ffmpeg falls back to `libx264` CPU encoding, which is
  5-10x slower.
- **Edge TTS is rate-limiting you.** Microsoft throttles to about 1
  request per second on the free service. A 35-segment script then
  takes 35+ seconds for the TTS phase alone. There is no workaround
  in the free tier; switch to a paid TTS backend if you hit this often.
- **Colab assigned a small VM.** Free tier GPUs are best-effort; if
  the assigned instance has less RAM, the encode may spill to swap.
  Rerun the cell and Colab will usually pick a better VM.

## Limitations of the free tier

- Colab free sessions cap at 12 hours and may disconnect when idle.
- GPU availability is not guaranteed; if no T4 is free, the notebook
  falls back to `libx264` software encoding (still 4-6x faster than a
  low-RAM Windows box because Colab has 12.7 GB free).
- The Edge TTS service is rate-limited; very long scripts may need
  to be split into chunks.
- Background videos up to 4K are supported but consume proportionally
  more memory during the loop step; 1080p is the sweet spot.

## Troubleshooting

If the render cell says `Render failed with exit code -12`, Colab ran
out of memory. Switch the runtime to a 25 GB RAM instance (Colab Pro)
or rerun the notebook - the failure is usually transient.